# Bitcoin StockMem: Event-Reflection Memory Framework

Reimplementation of the [StockMem paper](https://arxiv.org/abs/2512.02720) for **Bitcoin + ETH** price prediction.

- **LLM**: Gemini 2.5 Flash
- **Embeddings**: BGE-M3 (on GPU)
- **News**: CryptoPanic + RSS feeds
- **Price data**: Binance via ccxt

## 0. Setup

In [ ]:
# Install dependencies
!pip install -q google-generativeai sentence-transformers ccxt pycoingecko feedparser trafilatura scikit-learn pydantic python-dotenv

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/bitcoin_stockmem'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Storage directory: {DRIVE_ROOT}')

In [ ]:
# Clone or upload the project
# Option 1: If uploaded to Drive
PROJECT_DIR = '/content/bitcoin_stockmem'
if not os.path.exists(PROJECT_DIR):
    # Copy from Drive or upload manually
    !cp -r "{DRIVE_ROOT}/code" {PROJECT_DIR} 2>/dev/null || echo "Please upload bitcoin_stockmem/ to this Colab or to Drive"

# Add project to path
import sys
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Set API keys
# Option 1: Colab Secrets (recommended)
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    os.environ['CRYPTOPANIC_API_KEY'] = userdata.get('CRYPTOPANIC_API_KEY')
    print('API keys loaded from Colab Secrets')
except Exception:
    # Option 2: Manual input
    os.environ['GEMINI_API_KEY'] = ''  # <-- paste your key here
    os.environ['CRYPTOPANIC_API_KEY'] = ''  # <-- paste your key here
    print('Set API keys manually above')

assert os.environ.get('GEMINI_API_KEY'), 'GEMINI_API_KEY not set!'

In [ ]:
# Verify GPU and load embedding model
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

from embeddings.bge_m3 import encode
test_emb = encode(['Hello Bitcoin'])
print(f'Embedding dim: {test_emb.shape[1]}')

## 1. Configuration

In [ ]:
# Date ranges
TRAIN_START = '2025-01-01'
TRAIN_END = '2025-03-31'
TEST_START = '2025-04-01'
TEST_END = '2025-06-30'

# Override DB path to Drive
import config
config.DB_PATH = os.path.join(DRIVE_ROOT, 'stockmem.db')
print(f'Database: {config.DB_PATH}')
print(f'Assets: {config.ASSETS}')
print(f'Window size: {config.WINDOW_SIZE}')
print(f'Price threshold: ±{config.PRICE_THRESHOLD*100}%')

## 2. Data Collection

In [ ]:
from data.price_fetcher import fetch_daily_ohlcv
from data.label_generator import generate_labels, filter_tradable_days
from data.news_fetcher import fetch_all_news
from collections import defaultdict

# Fetch prices
prices = {}
for asset in config.ASSETS:
    df = fetch_daily_ohlcv(asset, TRAIN_START, TEST_END)
    df = generate_labels(df)
    prices[asset] = df
    print(f'{asset}: {len(df)} days, up={sum(df["label"]=="up")}, down={sum(df["label"]=="down")}, flat={sum(df["label"]=="flat")}')

# Display sample
prices['BTC'].tail(10)

In [ ]:
# Fetch news
all_news = fetch_all_news(TRAIN_START, TEST_END, scrape_bodies=True)
print(f'Total articles: {len(all_news)}')

# Group by date
news_by_date = defaultdict(list)
for a in all_news:
    news_by_date[a['date']].append(a)

print(f'Days with news: {len(news_by_date)}')
print(f'Avg articles/day: {len(all_news) / max(len(news_by_date), 1):.1f}')

## 3. Build Event Memory (Training Period)

Steps 1-3: Extract → Merge → Track for all training dates.
This is the most time-consuming step (~2-4 min per day at 15 RPM).

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

from llm.gemini_client import GeminiClient
from storage.database import get_connection
from evaluation.backtest import build_event_memory

# Init DB and client
get_connection()
client = GeminiClient()

# Training dates
train_dates = sorted(d for d in news_by_date.keys() if TRAIN_START <= d <= TRAIN_END)
train_news = {d: news_by_date[d] for d in train_dates}
print(f'Building event memory for {len(train_dates)} training days...')

build_event_memory(client, train_news, train_dates)
print('Event memory built!')

## 4. Build Reflection Memory (Training Period)

Step 4: Analyse causal relationships between events and price movements.

In [ ]:
from evaluation.backtest import build_reflection_memory

train_labels = {}
for asset, df in prices.items():
    train_df = df[(df['date'] >= TRAIN_START) & (df['date'] <= TRAIN_END)]
    train_labels[asset] = filter_tradable_days(train_df)
    print(f'{asset} training: {len(train_labels[asset])} tradable days')

build_reflection_memory(client, train_labels)
print('Reflection memory built!')

## 5. Run Backtest (Test Period)

Steps 5-6 + online learning: For each test day, predict → reveal truth → update memory.

In [ ]:
from evaluation.backtest import run_backtest

test_dates = sorted(d for d in news_by_date.keys() if TEST_START <= d <= TEST_END)
test_news = {d: news_by_date[d] for d in test_dates}

test_labels = {}
for asset, df in prices.items():
    test_df = df[(df['date'] >= TEST_START) & (df['date'] <= TEST_END)]
    test_labels[asset] = filter_tradable_days(test_df)
    print(f'{asset} test: {len(test_labels[asset])} tradable days')

print(f'Running backtest on {len(test_dates)} test days...')
results = run_backtest(client, test_news, test_labels, test_dates)

## 6. Results

In [ ]:
import pandas as pd

# Summary table
rows = []
for asset, metrics in results.items():
    rows.append({
        'Asset': asset,
        'Accuracy': metrics['accuracy'],
        'MCC': metrics['mcc'],
        'Total': metrics['total'],
        'Correct': metrics['correct'],
    })

results_df = pd.DataFrame(rows)
print('\n=== StockMem Results ===')
display(results_df)

# Overall average
if rows:
    avg_acc = sum(r['Accuracy'] for r in rows) / len(rows)
    avg_mcc = sum(r['MCC'] for r in rows) / len(rows)
    print(f'\nAverage ACC: {avg_acc:.4f}')
    print(f'Average MCC: {avg_mcc:.4f}')

In [ ]:
# Show sample predictions with reasoning
from storage.database import get_cursor

with get_cursor() as cur:
    cur.execute(
        '''SELECT date, asset, predicted_direction, actual_direction, reason
           FROM predictions
           WHERE actual_direction IS NOT NULL
           ORDER BY date DESC LIMIT 5'''
    )
    sample_preds = [dict(r) for r in cur.fetchall()]

for p in sample_preds:
    match = '✓' if p['predicted_direction'] == p['actual_direction'] else '✗'
    print(f"\n{match} {p['date']} {p['asset']}: predicted={p['predicted_direction']}, actual={p['actual_direction']}")
    print(f"   Reason: {p['reason'][:200]}...")